In [1]:
import pandas as pd
import polars as pl
import numpy as np
import matplotlib.pyplot as plt
import gc
from datetime import datetime
import altair as alt
alt.data_transformers.enable("vegafusion")

ohlcv = pd.read_csv("ohlcv_14nov_5dec_agg.csv")
df_l2 = pl.read_parquet("df_l2.parquet")
gc.collect()

20

In [2]:
print(ohlcv.sample(5))

                   ts      open      high       low     close        volume  \
134031  1763705100000  1.481300  1.483700  1.475600  1.481100  7.095120e+03   
140414  1763735100000  0.000004  0.000004  0.000004  0.000004  5.296000e+03   
63352   1763235300000  0.000005  0.000005  0.000005  0.000005  1.070509e+07   
100636  1763111100000  0.000005  0.000005  0.000005  0.000005  7.041726e+08   
149949  1764710700000  1.584300  1.588000  1.583400  1.586400  9.394100e+04   

             exch symbol  
134031       gate    SUI  
140414  gate_perp   PEPE  
63352          bg   PEPE  
100636       gate   PEPE  
149949  gate_perp    SUI  


In [3]:
print(df_l2.sample(2))

shape: (2, 44)
┌────────────┬──────────┬───────────┬───────────┬───┬──────────┬───────────┬───────────┬───────────┐
│ ts         ┆ exchange ┆ market_ty ┆ trading_p ┆ … ┆ bid_px_9 ┆ bid_sz_9  ┆ bid_px_10 ┆ bid_sz_10 │
│ ---        ┆ ---      ┆ pe        ┆ air       ┆   ┆ ---      ┆ ---       ┆ ---       ┆ ---       │
│ datetime[m ┆ str      ┆ ---       ┆ ---       ┆   ┆ f64      ┆ f64       ┆ f64       ┆ f64       │
│ s, UTC]    ┆          ┆ str       ┆ str       ┆   ┆          ┆           ┆           ┆           │
╞════════════╪══════════╪═══════════╪═══════════╪═══╪══════════╪═══════════╪═══════════╪═══════════╡
│ 2025-12-03 ┆ kucoin   ┆ spot      ┆ ETH-USDT  ┆ … ┆ 3021.2   ┆ 1.0575    ┆ 3021.14   ┆ 0.0355    │
│ 02:09:12   ┆          ┆           ┆           ┆   ┆          ┆           ┆           ┆           │
│ UTC        ┆          ┆           ┆           ┆   ┆          ┆           ┆           ┆           │
│ 2025-12-04 ┆ gate_io  ┆ spot      ┆ XRP-USDT  ┆ … ┆ 2.169    ┆ 14155.358 ┆

ohlcv-based feats that will be given from the environment

In [5]:
df_l2 = df_l2.with_columns(
    [
        (
            pl.col("ask_sz_1") * pl.col("ask_px_1")
            + pl.col("ask_sz_2") * pl.col("ask_px_2")
        ).alias("cum_usdt_ask_d2"),
        (
            pl.col("bid_sz_1") * pl.col("bid_px_1")
            + pl.col("bid_sz_2") * pl.col("bid_px_2")
        ).alias("cum_usdt_bid_d2"),

        (
            pl.col("ask_sz_1") * pl.col("ask_px_1")
            + pl.col("ask_sz_2") * pl.col("ask_px_2")
            + pl.col("ask_sz_3") * pl.col("ask_px_3")
            + pl.col("ask_sz_4") * pl.col("ask_px_4")
            + pl.col("ask_sz_5") * pl.col("ask_px_5")
        ).alias("cum_usdt_ask_d5"),
        (
            pl.col("bid_sz_1") * pl.col("bid_px_1")
            + pl.col("bid_sz_2") * pl.col("bid_px_2")
            + pl.col("bid_sz_3") * pl.col("bid_px_3")
            + pl.col("bid_sz_4") * pl.col("bid_px_4")
            + pl.col("bid_sz_5") * pl.col("bid_px_5")
        ).alias("cum_usdt_bid_d5"),
    ]
)

df_l2 = df_l2.with_columns([
        (pl.col("cum_usdt_bid_d2") + pl.col("cum_usdt_ask_d2")).alias(
            "cum_usdt_total_d2"
        ),

        (pl.col("cum_usdt_bid_d5") + pl.col("cum_usdt_ask_d5")).alias(
            "cum_usdt_total_d5"
        ),
])

df_l2 = df_l2.with_columns(
    [
        pl.when(pl.col("cum_usdt_total_d2") > 0)
        .then(pl.col("cum_usdt_bid_d2") / pl.col("cum_usdt_total_d2"))
        .otherwise(0.5)
        .alias("bid_ask_imbalance_d2"),

        pl.when(pl.col("cum_usdt_total_d5") > 0)
        .then(pl.col("cum_usdt_bid_d5") / pl.col("cum_usdt_total_d5"))
        .otherwise(0.5)
        .alias("bid_ask_imbalance_d5"),
    ]
)

df_l2 = df_l2.with_columns([
    pl.col("cum_usdt_total_d2").log1p().alias("log1p_cum_usdt_total_d2"),
    pl.col("cum_usdt_total_d5").log1p().alias("log1p_cum_usdt_total_d5")
])
df_l2 = df_l2.drop(["cum_usdt_ask_d2", "cum_usdt_bid_d2", "cum_usdt_ask_d5", "cum_usdt_bid_d5", "cum_usdt_total_d2", "cum_usdt_total_d5"])

In [6]:
df_l2["log1p_cum_usdt_total_d5"].skew(), df_l2["log1p_cum_usdt_total_d5"].kurtosis()

(-0.23662582538775045, -0.6883091711184286)

In [7]:
agg = (
    df_l2
    .select(
        "ts",
        "trading_pair",
        "market_type",
        "ask_px_1",
        "bid_px_1",
    )
    .with_columns(
        [
            pl.when((pl.col("market_type") == "spot") & (pl.col("ask_px_1") > 0))
            .then(pl.col("ask_px_1"))
            .otherwise(None)
            .alias("spot_best_ask"),
            pl.when((pl.col("market_type") == "perpetual") & (pl.col("ask_px_1") > 0))
            .then(pl.col("ask_px_1"))
            .otherwise(None)
            .alias("perp_best_ask"),
            pl.when((pl.col("market_type") == "spot") & (pl.col("bid_px_1") > 0))
            .then(pl.col("bid_px_1"))
            .otherwise(None)
            .alias("spot_best_bid"),
            pl.when((pl.col("market_type") == "perpetual") & (pl.col("bid_px_1") > 0))
            .then(pl.col("bid_px_1"))
            .otherwise(None)
            .alias("perp_best_bid"),
        ]
    )
    .group_by(["ts", "trading_pair"])
    .agg(
        [
            pl.col("spot_best_ask").mean().alias("spot_best_ask_mean"),
            pl.col("perp_best_ask").mean().alias("perp_best_ask_mean"),
            pl.col("spot_best_bid").mean().alias("spot_best_bid_mean"),
            pl.col("perp_best_bid").mean().alias("perp_best_bid_mean"),
        ]
    )
)

df_l2 = df_l2.join(agg, on=["ts", "trading_pair"], how="left")

del agg
gc.collect()

0

In [8]:
cols2drop = [i for i in df_l2.columns if ("px" in i or "sz" in i) and i not in ["ask_px_1", "bid_px_1"]]

In [9]:
df_l2 = df_l2.select(
    [i for i in df_l2.columns if i not in cols2drop]
).with_columns(
    [
        (
            pl.when(pl.col("market_type") == "spot")
            .then(1)
            .otherwise(0)
            .alias("is_spot")
        ),
        pl.when(pl.col("spot_best_bid_mean") > 0)
        .then(
            (pl.col("bid_px_1") - pl.col("spot_best_bid_mean"))
            / pl.col("spot_best_bid_mean")
        )
        .otherwise(None)
        .alias("dev_bid_from_spot_mean"),
        pl.when(pl.col("spot_best_ask_mean") > 0)
        .then(
            (pl.col("ask_px_1") - pl.col("spot_best_ask_mean"))
            / pl.col("spot_best_ask_mean")
        )
        .otherwise(None)
        .alias("dev_ask_from_spot_mean"),
        pl.when(pl.col("perp_best_bid_mean") > 0)
        .then(
            (pl.col("bid_px_1") - pl.col("perp_best_bid_mean"))
            / pl.col("perp_best_bid_mean")
        )
        .otherwise(None)
        .alias("dev_bid_from_perp_mean"),
        pl.when(pl.col("perp_best_ask_mean") > 0)
        .then(
            (pl.col("ask_px_1") - pl.col("perp_best_ask_mean"))
            / pl.col("perp_best_ask_mean")
        )
        .otherwise(None)
        .alias("dev_ask_from_perp_mean"),
    ]
).drop(["market_type", "spot_best_ask_mean", "perp_best_ask_mean", "spot_best_bid_mean", "perp_best_bid_mean"])

In [10]:
df_l2.columns

['ts',
 'exchange',
 'trading_pair',
 'ask_px_1',
 'bid_px_1',
 'bid_ask_imbalance_d2',
 'bid_ask_imbalance_d5',
 'log1p_cum_usdt_total_d2',
 'log1p_cum_usdt_total_d5',
 'is_spot',
 'dev_bid_from_spot_mean',
 'dev_ask_from_spot_mean',
 'dev_bid_from_perp_mean',
 'dev_ask_from_perp_mean']

In [11]:
df_l2.sample(2048).select(pl.col([pl.Int32, pl.Float64])).to_pandas().corr()

,ask_px_1,bid_px_1,bid_ask_imbalance_d2,bid_ask_imbalance_d5,log1p_cum_usdt_total_d2,log1p_cum_usdt_total_d5,is_spot,dev_bid_from_spot_mean,dev_ask_from_spot_mean,dev_bid_from_perp_mean,dev_ask_from_perp_mean
ask_px_1,1.000000,1.000000,0.020282,0.038764,0.359892,0.113500,0.004977,0.036635,0.068470,-0.033888,-0.073530
bid_px_1,1.000000,1.000000,0.020282,0.038764,0.359892,0.113499,0.004977,0.036634,0.068469,-0.033888,-0.073531
bid_ask_imbalance_d2,0.020282,0.020282,1.000000,0.689638,0.015956,-0.003060,-0.001869,-0.036989,-0.039777,-0.093693,-0.095166
bid_ask_imbalance_d5,0.038764,0.038764,0.689638,1.000000,0.017793,-0.016828,0.033325,0.006541,0.007649,-0.032915,-0.031183
log1p_cum_usdt_total_d2,0.359892,0.359892,0.015956,0.017793,1.000000,0.886299,-0.332585,-0.217304,-0.162446,-0.444168,-0.329895
log1p_cum_usdt_total_d5,0.113500,0.113499,-0.003060,-0.016828,0.886299,1.000000,-0.368885,-0.287282,-0.214515,-0.475766,-0.334999
is_spot,0.004977,0.004977,-0.001869,0.033325,-0.332585,-0.368885,1.000000,0.835010,0.868739,0.809862,0.848584
dev_bid_from_spot_mean,0.036635,0.036634,-0.036989,0.006541,-0.217304,-0.287282,0.835010,1.000000,0.956867,0.849445,0.834908
dev_ask_from_spot_mean,0.068470,0.068469,-0.039777,0.007649,-0.162446,-0.214515,0.868739,0.956867,1.000000,0.830629,0.881846
dev_bid_from_perp_mean,-0.033888,-0.033888,-0.093693,-0.032915,-0.444168,-0.475766,0.809862,0.849445,0.830629,1.000000,0.954519


In [12]:
df_l2.write_parquet("df_l2_ready.parquet")

---

~~price/vol-sensitive cols normalization (using (x- mu)/std)~~

### final prep

In [1]:
import pandas as pd
import polars as pl
import numpy as np
import matplotlib.pyplot as plt
import gc
from datetime import datetime
import altair as alt
import joblib
alt.data_transformers.enable("vegafusion")
pd.set_option("display.max_columns", None)

from collections import defaultdict

df_ohlcv = pd.read_csv("ohlcv_14nov_5dec_agg.csv")
df_l2 = pl.read_parquet("df_l2_ready.parquet")
gc.collect()

20

rolling feats

In [2]:
imb_window = 50 # 5 sec
liq_window = 100 # 10 sec

df_l2 = (
    df_l2
    .sort(["trading_pair", "exchange", "ts"])
    .with_columns([
        pl.col("bid_ask_imbalance_d5")
          .rolling_mean(window_size=imb_window, min_samples=imb_window)
          .over(["trading_pair", "exchange"])
          .alias(f"bid_ask_imbalance_d5_ma_{imb_window}"),

        pl.col("log1p_cum_usdt_total_d5")
          .rolling_mean(window_size=liq_window, min_samples=liq_window)
          .over(["trading_pair", "exchange"])
          .alias(f"log1p_cum_usdt_total_d5_ma_{liq_window}"),
    ])
)

In [3]:
print(df_l2.shape)
df_l2 = df_l2.drop_nulls([f"bid_ask_imbalance_d5_ma_{imb_window}", f"log1p_cum_usdt_total_d5_ma_{liq_window}"])
print(df_l2.shape)

(125189890, 16)
(125187415, 16)


In [4]:
df_l2.null_count()
# will clear them at the end (in torch dataset!!! now it will leak all the memory)

ts,exchange,trading_pair,ask_px_1,bid_px_1,bid_ask_imbalance_d2,bid_ask_imbalance_d5,log1p_cum_usdt_total_d2,log1p_cum_usdt_total_d5,is_spot,dev_bid_from_spot_mean,dev_ask_from_spot_mean,dev_bid_from_perp_mean,dev_ask_from_perp_mean,bid_ask_imbalance_d5_ma_50,log1p_cum_usdt_total_d5_ma_100
u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32
0,0,0,0,0,0,0,0,0,0,45,45,55,55,0,0


In [5]:
df_l2["bid_ask_imbalance_d5"].plot.hist()

alt.Chart(...)

In [6]:
df_l2 = df_l2.rename({"ask_px_1": "best_ask_raw", "bid_px_1": "best_bid_raw"})

df_l2_train = df_l2.filter((pl.col("ts") < pd.to_datetime("2025-12-05 03:00:00", utc=True)) & (pl.col("ts") > pd.to_datetime("2025-11-30 23:00:00", utc=True))) # earlier has gaps (start from 2025-11-30 00:32:00 if ok with 4min x2 gaps)
df_l2_test = df_l2.filter((pl.col("ts") >= pd.to_datetime("2025-12-05 03:00:00", utc=True)) & (pl.col("ts") < pd.to_datetime("2025-12-05 19:35:00", utc=True))) # ohlcv data ends at 19:35, 5 dec

df_l2_train.shape, df_l2_test.shape, df_l2_test.shape[0]/df_l2.shape[0]

((88640410, 16), (14637130, 16), 0.11692173690142894)

In [7]:
# coin_statistics = df_l2_train.group_by("trading_pair").agg(
#     [
#         pl.col("ask_px_1").mean().alias("price_mean"),
#         pl.col("ask_px_1").std().alias("price_std"),
#     ]
# ).to_pandas()

# mu_std_mapping = coin_statistics.set_index("trading_pair").to_dict(orient="index")

# mu_std_mapping

In [8]:
# norm_df = pl.DataFrame(
#     {
#         "trading_pair": list(mu_std_mapping.keys()),
#         "price_mean": [v["price_mean"] for v in mu_std_mapping.values()],
#         "price_std": [v["price_std"] for v in mu_std_mapping.values()],
#     }
# )

# norm_df

In [9]:
# df_l2_train = (
#     df_l2_train.join(norm_df, on="trading_pair", how="left")
#     .with_columns(
#         [
#             ((pl.col("ask_px_1") - pl.col("price_mean")) / pl.col("price_std")).alias("ask_px_1_norm"),
#             ((pl.col("bid_px_1") - pl.col("price_mean")) / pl.col("price_std")).alias("bid_px_1_norm"),
#         ]
#     )
#     .drop(["price_mean", "price_std"])
# )

# df_l2_test = (
#     df_l2_test.join(norm_df, on="trading_pair", how="left")
#     .with_columns(
#         [
#             ((pl.col("ask_px_1") - pl.col("price_mean")) / pl.col("price_std")).alias("ask_px_1_norm"),
#             ((pl.col("bid_px_1") - pl.col("price_mean")) / pl.col("price_std")).alias("bid_px_1_norm"),
#         ]
#     )
#     .drop(["price_mean", "price_std"])
# )

In [10]:
df_l2_test.sample(5)

ts,exchange,trading_pair,best_ask_raw,best_bid_raw,bid_ask_imbalance_d2,bid_ask_imbalance_d5,log1p_cum_usdt_total_d2,log1p_cum_usdt_total_d5,is_spot,dev_bid_from_spot_mean,dev_ask_from_spot_mean,dev_bid_from_perp_mean,dev_ask_from_perp_mean,bid_ask_imbalance_d5_ma_50,log1p_cum_usdt_total_d5_ma_100
"datetime[ms, UTC]",str,str,f64,f64,f64,f64,f64,f64,i32,f64,f64,f64,f64,f64,f64
2025-12-05 19:06:07 UTC,"""gate_io""","""XRP-USDT""",2.02,2.019,0.294659,0.449808,11.768442,12.99635,1,0.000031,0.000343,0.000719,0.001165,0.457524,12.966339
2025-12-05 12:40:59.800 UTC,"""gate_io""","""SUI-USDT""",1.6232,1.6231,0.477641,0.731758,7.12925,9.189972,1,0.000021,0.000021,0.00074,0.00074,0.662424,9.060592
2025-12-05 14:30:05.800 UTC,"""bitget""","""XRP-USDT""",2.0639,2.0638,0.520336,0.578734,9.631961,10.840041,1,0.000118,-0.000013,0.000582,0.000582,0.373746,10.857535
2025-12-05 04:55:29.500 UTC,"""bitget_perpetual""","""SOL-USDT""",138.581,138.58,0.059761,0.490969,11.136633,11.825908,0,-0.000168,-0.000233,0.00018,0.000148,0.530532,11.581761
2025-12-05 18:46:02.800 UTC,"""gate_io_perpetual""","""XRP-USDT""",2.0191,2.019,0.805468,0.682178,10.494639,10.705544,0,-0.000571,-0.000704,0.000025,0.000025,0.542211,10.3295


---

5-min ohlcv based

In [11]:
print(df_ohlcv.head(5))

              ts    open    high     low   close    volume  exch symbol
0  1763078700000  144.93  145.02  144.34  144.61  1743.639  gate    SOL
1  1763079000000  144.70  145.03  144.49  144.92  1523.220  gate    SOL
2  1763079300000  144.92  144.92  143.47  144.35  2076.408  gate    SOL
3  1763079600000  144.35  144.47  144.04  144.30   651.264  gate    SOL
4  1763079900000  144.26  144.78  144.17  144.21  1397.935  gate    SOL


In [12]:
df_ohlcv.sort_values(["ts", "symbol"])

,ts,open,high,low,close,volume,exch,symbol
12566,1763078700000,3225.94000,3229.02000,3213.01000,3214.51000,1.246849e+03,kucoin,ETH
69113,1763078700000,3226.18000,3229.11000,3212.88000,3214.42000,4.692175e+02,bg,ETH
87962,1763078700000,3223.30000,3227.54000,3211.10000,3212.81000,1.553243e+06,gate_perp,ETH
106811,1763078700000,3225.74000,3229.21000,3213.00000,3214.63000,1.001781e+03,gate,ETH
113094,1763078700000,3225.54000,3228.86000,3212.13000,3213.67000,4.499000e+03,bg_perp,ETH
...,...,...,...,...,...,...,...,...
56546,1764963300000,2.03340,2.03800,2.03240,2.03800,5.126085e+04,bg,XRP
81678,1764963300000,2.03300,2.03900,2.03200,2.03800,1.282956e+05,gate,XRP
100527,1764963300000,2.03298,2.03816,2.03295,2.03816,2.475073e+05,kucoin,XRP
125659,1764963300000,2.03200,2.03670,2.03140,2.03660,4.179200e+04,gate_perp,XRP


In [13]:
df_ohlcv["ts"] = pd.to_datetime(df_ohlcv["ts"], unit="ms", utc=True)
df_ohlcv = df_ohlcv.sort_values(["symbol", "ts"])

def build_ohlcv_features(g):
    g = g.set_index("ts").sort_index()

    # 5min candles agg
    ohlcv = g[["open", "high", "low", "close", "volume"]].resample("5min").agg({
        "open": "mean",
        "high": "max",
        "low": "min",
        "close": "mean",
        "volume": "sum"
    }).dropna()

    # inter-exchange variance
    disp = g["close"].resample("5min").agg(["mean", "std", "max", "min"])
    disp["cross_close_std_5m"] = disp["std"]
    disp["cross_close_range_5m"] = (disp["max"] - disp["min"]) / disp["mean"]

    ohlcv = ohlcv.join(disp[["cross_close_std_5m", "cross_close_range_5m"]], how="left")

    ohlcv["ret_1"] = ohlcv["close"].pct_change(1)
    ohlcv["ret_3"] = ohlcv["close"].pct_change(3)
    ohlcv["ret_6"] = ohlcv["close"].pct_change(6)

    ohlcv["ma_3"] = ohlcv["close"].rolling(3).mean()
    ohlcv["ma_6"] = ohlcv["close"].rolling(6).mean()
    ohlcv["ma_12"] = ohlcv["close"].rolling(12).mean()

    ohlcv["slope"] = (ohlcv["ma_3"] - ohlcv["ma_12"]) / ohlcv["ma_12"]

    ohlcv["range"] = (ohlcv["high"] - ohlcv["low"]) / ohlcv["close"]
    ohlcv["range_mean_6"] = ohlcv["range"].rolling(6).mean()

    ohlcv["vol_6"] = ohlcv["ret_1"].rolling(6).std()
    ohlcv["vol_12"] = ohlcv["ret_1"].rolling(12).std()

    ohlcv["atr"] = (ohlcv["high"] - ohlcv["low"]).ewm(span=6, adjust=False).mean()

    ohlcv["vol_norm"] = ohlcv["volume"] / ohlcv["volume"].rolling(6).mean()
    ohlcv["vol_trend"] = ohlcv["volume"].pct_change(1)

    ohlcv["ret_1w"] = ohlcv["close"].pct_change(2016)
    ohlcv["vol_1w"] = ohlcv["ret_1"].rolling(2016).std()

    ohlcv["ret_4h"] = ohlcv["close"].pct_change(48)
    ohlcv["vol_4h"] = ohlcv["ret_1"].rolling(48).std()

    ohlcv["vol_1d"] = ohlcv["ret_1"].rolling(288).std()

    roll_max_1d = ohlcv["close"].rolling(288).max()
    roll_min_1d = ohlcv["close"].rolling(288).min()
    ohlcv["pos_1d"] = (ohlcv["close"] - roll_min_1d) / (roll_max_1d - roll_min_1d)

    return ohlcv

ohlcv_df_ready = (
    df_ohlcv.groupby("symbol", group_keys=True)
            .apply(build_ohlcv_features)
            .reset_index()
)

ohlcv_df_ready = ohlcv_df_ready.sort_values(["ts", "symbol"])

/tmp/ipykernel_433540/2219155957.py:60: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(build_ohlcv_features)


find stats for ma_* normalization, fair way

In [14]:
close_stats = ohlcv_df_ready[ohlcv_df_ready["ts"] < pd.to_datetime("2025-12-05 03:00:00", utc=True)].groupby("symbol")["close"].agg(["mean", "std"]).to_dict(orient="index")
close_stats

{'ETH': {'mean': 2990.652190695381, 'std': 143.06248920118585},
 'PEPE': {'mean': 4.6053773664310375e-06, 'std': 3.178496138072255e-07},
 'SOL': {'mean': 136.60662045043566, 'std': 5.534221459589297},
 'SUI': {'mean': 1.5590642807825086, 'std': 0.13856420979196957},
 'XRP': {'mean': 2.146479424626007, 'std': 0.10074643192496047}}

In [15]:
print(ohlcv_df_ready.ma_3.min(), ohlcv_df_ready.ma_3.max())

ohlcv_df_ready["ma_3"] = ohlcv_df_ready.apply(lambda x: (x["ma_3"] - close_stats[x["symbol"]]["mean"]) / close_stats[x["symbol"]]["std"], axis=1)
ohlcv_df_ready["ma_6"] = ohlcv_df_ready.apply(lambda x: (x["ma_6"] - close_stats[x["symbol"]]["mean"]) / close_stats[x["symbol"]]["std"], axis=1)
ohlcv_df_ready["ma_12"] = ohlcv_df_ready.apply(lambda x: (x["ma_12"] - close_stats[x["symbol"]]["mean"]) / close_stats[x["symbol"]]["std"], axis=1)

print(ohlcv_df_ready.ma_3.min(), ohlcv_df_ready.ma_3.max())

3.969526666666667e-06 3243.1186666666667
-3.0744919236780746 2.595910972967408


In [16]:
ohlcv_df_ready.describe()

,open,high,low,close,volume,cross_close_std_5m,cross_close_range_5m,ret_1,ret_3,ret_6,ma_3,ma_6,ma_12,slope,range,range_mean_6,vol_6,vol_12,atr,vol_norm,vol_trend,ret_1w,vol_1w,ret_4h,vol_4h,vol_1d,pos_1d
count,31415.000000,31415.000000,31415.000000,31415.000000,3.141500e+04,3.141500e+04,31415.000000,31410.000000,31400.000000,31385.000000,31405.000000,31390.000000,31360.000000,31360.000000,31415.000000,31390.000000,31385.000000,31355.000000,3.141500e+04,31390.000000,31410.000000,21335.000000,21335.000000,31175.000000,31175.000000,29975.000000,29980.000000
mean,627.010516,628.174383,625.762548,627.003696,9.105367e+09,1.617591e-01,0.000767,-0.000017,-0.000051,-0.000105,0.002829,0.002496,0.001867,-0.000094,0.004480,0.004480,0.002414,0.002508,2.412420e+00,1.011791,0.285255,-0.019179,0.002813,-0.000768,0.002634,0.002737,0.494518
std,1186.770537,1188.954178,1184.423980,1186.757113,4.169631e+10,3.116694e-01,0.000280,0.002908,0.004943,0.006998,0.990228,0.989275,0.987395,0.004338,0.002851,0.002098,0.001640,0.001478,5.145964e+00,0.550234,1.686763,0.094928,0.000623,0.020700,0.001227,0.000885,0.304531
min,0.000004,0.000004,0.000004,0.000004,4.578818e+03,8.944272e-11,0.000045,-0.060181,-0.062919,-0.070176,-3.074492,-2.941074,-2.788119,-0.040902,0.000843,0.001268,0.000058,0.000323,8.168678e-09,0.055097,-0.958958,-0.265379,0.001768,-0.101396,0.000672,0.000995,0.000000
25%,1.486610,1.491250,1.481950,1.486570,1.477639e+05,5.049752e-04,0.000579,-0.001398,-0.002429,-0.003454,-0.728653,-0.731594,-0.730036,-0.002087,0.002737,0.003060,0.001377,0.001547,5.368619e-03,0.649416,-0.335107,-0.087199,0.002174,-0.010000,0.001754,0.002157,0.221768
50%,2.177688,2.181200,2.174400,2.177766,4.904582e+05,7.588939e-04,0.000714,-0.000048,-0.000114,-0.000128,0.167403,0.166208,0.163012,-0.000066,0.003771,0.003961,0.002026,0.002153,1.009994e-02,0.883163,-0.036434,-0.011420,0.002929,-0.000722,0.002325,0.002637,0.504427
75%,141.102900,141.380000,140.817000,141.099100,1.806636e+06,4.605649e-02,0.000901,0.001290,0.002185,0.003208,0.726162,0.727830,0.725622,0.001952,0.005391,0.005326,0.002987,0.003056,6.662336e-01,1.223161,0.439036,0.054564,0.003187,0.008636,0.003284,0.003180,0.759929
max,3247.906000,3257.390000,3243.470000,3247.976000,4.513382e+12,2.166811e+00,0.005359,0.039410,0.058573,0.061450,2.595911,2.562971,2.526004,0.040740,0.110989,0.032935,0.028077,0.019768,4.639685e+01,5.609582,155.997520,0.201882,0.003969,0.163121,0.010167,0.006067,1.000000


In [17]:
ohlcv_df_ready = ohlcv_df_ready[ohlcv_df_ready["ts"] >= pd.to_datetime("2025-11-29 22:50:00", utc=True)].reset_index(drop=True)
ohlcv_df_ready.drop(columns=["open", "high", "low", "close", "volume"], inplace=True)
ohlcv_df_ready.isna().sum().sum() == 0

True

In [18]:
train_ts = df_l2_train["ts"].to_pandas().sort_values().unique()
test_ts = df_l2_test["ts"].to_pandas().sort_values().unique()

ohlcv_ts = ohlcv_df_ready.ts.sort_values().unique()

# train_ts = pd.to_datetime(df_l2_train["ts"].unique().sort_values().to_numpy(), utc=True)

---

flatten (pivot)

In [21]:
df_l2_train = df_l2_train.with_columns([
    pl.col("dev_bid_from_perp_mean").fill_null(strategy="zero"),
    pl.col("dev_ask_from_perp_mean").fill_null(strategy="zero"),
])

venue_order = [
    "bitget",
    "gate_io",
    "kucoin",
    "gate_io_perpetual",
    "bitget_perpetual",
]

feature_cols = [i for i in df_l2_train.columns[3:] if i != "is_spot"]

df_l2_train = df_l2_train.with_columns([
    pl.col("exchange").alias("venue")
])
df_l2_test = df_l2_test.with_columns([
    pl.col("exchange").alias("venue")
])

cols_order = []

for i in venue_order:
    for j in feature_cols:
        cols_order.append(f"{j}_{i}")

len(cols_order)

60

In [22]:
train_df_wide = df_l2_train.pivot(
    index=["ts", "trading_pair"],
    on="venue",
    values=feature_cols,
    aggregate_function="first"
)

test_df_wide = df_l2_test.pivot(
    index=["ts", "trading_pair"],
    on="venue",
    values=feature_cols,
    aggregate_function="first"
)

In [23]:
train_df_wide = train_df_wide.fill_null(strategy="zero")
test_df_wide = test_df_wide.fill_null(strategy="zero")

In [24]:
del df_l2_train, df_l2_test, df_l2
gc.collect()

0

---

calc fast-lookup indexes (1->M - ohlcv_df_ready->df_l2...)

In [25]:
match_res_train = np.searchsorted(ohlcv_ts, train_ts, side="left") - 2
assert len(match_res_train) == len(train_ts)

match_res_test = np.searchsorted(ohlcv_ts, test_ts, side="left") - 2
assert len(match_res_test) == len(test_ts)

In [26]:
match_res_train

array([ 289,  289,  289, ..., 1488, 1488, 1488])

In [27]:
train_idx_ohlcv = pl.DataFrame({"ts_ms": list(map(lambda x: int(x.timestamp()*1000), train_ts.to_pydatetime())), "idx_to_get_feats": match_res_train})
test_idx_ohlcv = pl.DataFrame({"ts_ms": list(map(lambda x: int(x.timestamp()*1000), test_ts.to_pydatetime())), "idx_to_get_feats": match_res_test})

In [29]:
train_df_wide = train_df_wide.with_columns([
    (pl.col("ts").cast(pl.Datetime(time_unit="ms")).cast(pl.Int64)).alias("ts_ms")
])
test_df_wide = test_df_wide.with_columns([
    (pl.col("ts").cast(pl.Datetime(time_unit="ms")).cast(pl.Int64)).alias("ts_ms")
])

In [30]:
train_df_wide = train_df_wide.join(train_idx_ohlcv, on="ts_ms", how="left")
test_df_wide = test_df_wide.join(test_idx_ohlcv, on="ts_ms", how="left")

train_df_wide = train_df_wide.drop("ts_ms")
test_df_wide = test_df_wide.drop("ts_ms")

In [33]:
ohlcv_df_ready_eth = ohlcv_df_ready[ohlcv_df_ready["symbol"] == "ETH"].sort_values("ts").reset_index(drop=True)
ohlcv_df_ready_sol = ohlcv_df_ready[ohlcv_df_ready["symbol"] == "SOL"].sort_values("ts").reset_index(drop=True)
ohlcv_df_ready_pepe = ohlcv_df_ready[ohlcv_df_ready["symbol"] == "PEPE"].sort_values("ts").reset_index(drop=True)
ohlcv_df_ready_sui = ohlcv_df_ready[ohlcv_df_ready["symbol"] == "SUI"].sort_values("ts").reset_index(drop=True)
ohlcv_df_ready_xrp = ohlcv_df_ready[ohlcv_df_ready["symbol"] == "XRP"].sort_values("ts").reset_index(drop=True)

test logic:

In [45]:
ohlcv_df_ready_pepe.iloc[1486]

symbol                                       PEPE
ts                      2025-12-05 02:40:00+00:00
cross_close_std_5m                            0.0
cross_close_range_5m                     0.001257
ret_1                                   -0.001756
ret_3                                    0.000101
ret_6                                    0.002629
ma_3                                     0.539257
ma_6                                     0.530647
ma_12                                    0.510407
slope                                    0.001923
range                                    0.003351
range_mean_6                             0.002744
vol_6                                    0.001505
vol_12                                   0.001516
atr                                           0.0
vol_norm                                 2.186313
vol_trend                                 2.95258
ret_1w                                   0.030731
vol_1w                                   0.002749


In [44]:
train_df_wide[-6000-1000]

ts,trading_pair,best_ask_raw_bitget,best_ask_raw_bitget_perpetual,best_ask_raw_gate_io,best_ask_raw_gate_io_perpetual,best_ask_raw_kucoin,best_bid_raw_bitget,best_bid_raw_bitget_perpetual,best_bid_raw_gate_io,best_bid_raw_gate_io_perpetual,best_bid_raw_kucoin,bid_ask_imbalance_d2_bitget,bid_ask_imbalance_d2_bitget_perpetual,bid_ask_imbalance_d2_gate_io,bid_ask_imbalance_d2_gate_io_perpetual,bid_ask_imbalance_d2_kucoin,bid_ask_imbalance_d5_bitget,bid_ask_imbalance_d5_bitget_perpetual,bid_ask_imbalance_d5_gate_io,bid_ask_imbalance_d5_gate_io_perpetual,bid_ask_imbalance_d5_kucoin,log1p_cum_usdt_total_d2_bitget,log1p_cum_usdt_total_d2_bitget_perpetual,log1p_cum_usdt_total_d2_gate_io,log1p_cum_usdt_total_d2_gate_io_perpetual,log1p_cum_usdt_total_d2_kucoin,log1p_cum_usdt_total_d5_bitget,log1p_cum_usdt_total_d5_bitget_perpetual,log1p_cum_usdt_total_d5_gate_io,log1p_cum_usdt_total_d5_gate_io_perpetual,log1p_cum_usdt_total_d5_kucoin,dev_bid_from_spot_mean_bitget,dev_bid_from_spot_mean_bitget_perpetual,dev_bid_from_spot_mean_gate_io,dev_bid_from_spot_mean_gate_io_perpetual,dev_bid_from_spot_mean_kucoin,dev_ask_from_spot_mean_bitget,dev_ask_from_spot_mean_bitget_perpetual,dev_ask_from_spot_mean_gate_io,dev_ask_from_spot_mean_gate_io_perpetual,dev_ask_from_spot_mean_kucoin,dev_bid_from_perp_mean_bitget,dev_bid_from_perp_mean_bitget_perpetual,dev_bid_from_perp_mean_gate_io,dev_bid_from_perp_mean_gate_io_perpetual,dev_bid_from_perp_mean_kucoin,dev_ask_from_perp_mean_bitget,dev_ask_from_perp_mean_bitget_perpetual,dev_ask_from_perp_mean_gate_io,dev_ask_from_perp_mean_gate_io_perpetual,dev_ask_from_perp_mean_kucoin,bid_ask_imbalance_d5_ma_50_bitget,bid_ask_imbalance_d5_ma_50_bitget_perpetual,bid_ask_imbalance_d5_ma_50_gate_io,bid_ask_imbalance_d5_ma_50_gate_io_perpetual,bid_ask_imbalance_d5_ma_50_kucoin,log1p_cum_usdt_total_d5_ma_100_bitget,log1p_cum_usdt_total_d5_ma_100_bitget_perpetual,log1p_cum_usdt_total_d5_ma_100_gate_io,log1p_cum_usdt_total_d5_ma_100_gate_io_perpetual,log1p_cum_usdt_total_d5_ma_100_kucoin,idx_to_get_feats
"datetime[ms, UTC]",str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,i64
2025-12-05 02:49:12.800 UTC,"""XRP-USDT""",2.1076,2.1065,2.108,2.1065,2.10762,2.1075,2.1064,2.107,2.1064,2.10761,0.231613,0.589518,0.57764,0.893102,0.443354,0.327781,0.627163,0.533259,0.743629,0.599686,9.851711,11.494343,11.919868,11.43626,8.947194,11.269222,12.990081,13.243796,11.846648,9.97089,0.000062,-0.00046,-0.000176,-0.00046,0.000114,-0.000066,-0.000588,0.000123,-0.000588,-0.000057,0.000522,0.0,0.000285,0.0,0.000574,0.000522,0.0,0.000712,0.0,0.000532,0.335671,0.616589,0.55376,0.621338,0.523152,11.200463,12.97183,13.287834,11.569336,9.87548,1486


---

dump everything in a correct way:

In [46]:
ohlcv_dfs = {
    "ETH": ohlcv_df_ready_eth,
    "SOL": ohlcv_df_ready_sol,
    "PEPE": ohlcv_df_ready_pepe,
    "SUI": ohlcv_df_ready_sui,
    "XRP": ohlcv_df_ready_xrp,
}

joblib.dump(ohlcv_dfs, "ohlcv_dfs_FINAL.pkl")

['ohlcv_dfs_FINAL.pkl']

In [ ]:
import json

with open("df_l2_cols_order.json", "w") as f:
    json.dump(["ts", "trading_pair"] + cols_order + ["idx_to_get_feats"], f, indent=4)

In [47]:
train_df_wide.write_parquet("df_l2_train_FINAL_WIDE.parquet")
test_df_wide.write_parquet("df_l2_test_FINAL_WIDE.parquet")

In [43]:
# df_l2_train.with_columns([
#     pl.col(col).cast(pl.Float32)
#     for col, dtype in df_l2_train.schema.items()
#     if dtype == pl.Float64
# ]) # strange results, ll do later

In [50]:
ohlcv_df_ready.select_dtypes(include=[int, float]).corr()

,cross_close_std_5m,cross_close_range_5m,ret_1,ret_3,ret_6,ma_3,ma_6,ma_12,slope,range,range_mean_6,vol_6,vol_12,atr,vol_norm,vol_trend,ret_1w,vol_1w,ret_4h,vol_4h,vol_1d,pos_1d
cross_close_std_5m,1.000000,-0.291594,0.002700,0.012101,0.017383,0.150050,0.149434,0.148327,0.022072,-0.120893,-0.164395,-0.120702,-0.145587,0.867541,0.006310,-0.020376,0.129752,-0.478489,0.032638,-0.195834,-0.265137,0.173225
cross_close_range_5m,-0.291594,1.000000,0.002472,0.029349,0.043953,-0.161543,-0.162960,-0.164964,0.042026,0.241672,0.288315,0.191448,0.224710,-0.286676,0.001888,0.025238,0.020045,0.508918,-0.002984,0.257318,0.301443,-0.081258
ret_1,0.002700,0.002472,1.000000,0.584251,0.427490,-0.005626,-0.013363,-0.020492,0.216184,0.033929,0.010485,0.014027,0.002958,0.000235,-0.001952,0.026104,0.009329,-0.004926,0.173680,-0.003220,0.016027,0.147196
ret_3,0.012101,0.029349,0.584251,1.000000,0.720073,0.015487,-0.010336,-0.028002,0.618906,0.040808,0.028311,0.047553,0.018186,0.000539,0.003211,0.007829,0.015656,-0.008939,0.299208,-0.003975,0.028799,0.241909
ret_6,0.017383,0.043953,0.427490,0.720073,1.000000,0.040027,0.012982,-0.022682,0.893416,0.014812,0.027144,0.058453,0.026142,-0.005879,-0.008138,-0.002684,0.023087,-0.012783,0.417617,-0.007169,0.038491,0.317768
ma_3,0.150050,-0.161543,-0.005626,0.015487,0.040027,1.000000,0.999428,0.997293,0.062933,-0.038673,-0.050493,0.020139,0.024369,0.134986,-0.000264,0.019758,0.697976,0.046517,0.189204,0.035200,0.130892,0.444801
ma_6,0.149434,-0.162960,-0.013363,-0.010336,0.012982,0.999428,1.000000,0.998811,0.035982,-0.038655,-0.051128,0.018548,0.023597,0.135167,0.000286,0.020176,0.697691,0.046900,0.178162,0.035453,0.129734,0.435801
ma_12,0.148327,-0.164964,-0.020492,-0.028002,-0.022682,0.997293,0.998811,1.000000,-0.007186,-0.038042,-0.051084,0.016224,0.021943,0.135899,0.000835,0.020327,0.696841,0.047703,0.154833,0.035792,0.127501,0.418950
slope,0.022072,0.042026,0.216184,0.618906,0.893416,0.062933,0.035982,-0.007186,1.000000,0.003093,0.018012,0.056027,0.033086,-0.007593,-0.009560,-0.009248,0.031441,-0.016942,0.501040,-0.009326,0.042872,0.359525
range,-0.120893,0.241672,0.033929,0.040808,0.014812,-0.038673,-0.038655,-0.038042,0.003093,1.000000,0.727583,0.653758,0.637913,0.039347,0.414516,0.350416,0.000720,0.277280,0.047425,0.504732,0.348075,-0.019952
